# 9강 적용하기 — Edge AI 모델 경량화 (Quantization)

설비 이상탐지 모델을 Edge Device에 배포하려고 합니다. 원본 모델은 정확도는 높지만
크기가 커서 추론 시간이 깁니다. Weight를 낮은 정밀도(FP16)로 변환해 크기를 줄이고,
정확도가 어느 정도 유지되는지 확인합니다.

이 실습은 EdgeX Foundry와 무관합니다 — 모델 학습·양자화는 플랫폼이 아니라
순수 ML 도구(scikit-learn/numpy)의 영역이라 별도의 가벼운 환경에서 진행합니다.

In [ ]:
import os
import pickle
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. 실습 데이터 생성

In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=12,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 2~3. 원본 모델 학습 및 평가

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

original_pred = model.predict(X_test)
original_accuracy = accuracy_score(y_test, original_pred)
print("Original Accuracy :", round(original_accuracy, 4))

## 4. 원본 모델 저장 및 크기 확인

In [ ]:
with open("original_model.pkl", "wb") as f:
    pickle.dump(model, f)
original_size = os.path.getsize("original_model.pkl")
print("Original Model Size :", original_size, "bytes")

## 5. Quantization 개념 적용 — FP64 Weight를 FP16으로 변환

In [ ]:
quantized_coef = model.coef_.astype(np.float16)
quantized_intercept = model.intercept_.astype(np.float16)
print("변환 후 dtype:", quantized_coef.dtype)

## 6. 경량 모델 저장

In [ ]:
quantized_model = {"coef": quantized_coef, "intercept": quantized_intercept}
with open("quantized_model.pkl", "wb") as f:
    pickle.dump(quantized_model, f)
quantized_size = os.path.getsize("quantized_model.pkl")
print("Quantized Model Size :", quantized_size, "bytes")

## 7. Quantized Weight로 예측

In [ ]:
score = X_test @ quantized_coef.T + quantized_intercept
quantized_pred = (score.ravel() >= 0).astype(int)
quantized_accuracy = accuracy_score(y_test, quantized_pred)
print("Quantized Accuracy :", round(quantized_accuracy, 4))

## 8. 결과 비교

In [ ]:
print("\n==========================")
print("Optimization Result")
print("==========================")
print("Original Accuracy  :", round(original_accuracy, 4))
print("Quantized Accuracy :", round(quantized_accuracy, 4))
print("Original Size      :", original_size, "bytes")
print("Quantized Size     :", quantized_size, "bytes")
reduction = (1 - quantized_size / original_size) * 100
print("Size Reduction     :", round(reduction, 2), "%")

## 심화 — 순수 정밀도 효과만 따로 떼어보기

위 8번의 감소율(약 71~72%)에는 두 가지가 섮여 있습니다.
1. **정밀도 감소 효과** (FP64→8바이트 → FP16 2바이트, 이론상 4배/75%)
2. **메타데이터 제거 효과** (sklearn 객체 전체 대신 coef/intercept만 저장)

순수하게 정밀도 효과만 보려면, 원본도 동일한 형태(딕셔너리)로 저장해 비교해야 합니다.

In [ ]:
# 공정한 비교: 원본도 동일하게 dict로만 저장
original_dict = {"coef": model.coef_.astype(np.float64), "intercept": model.intercept_.astype(np.float64)}
with open("original_weights_only.pkl", "wb") as f:
    pickle.dump(original_dict, f)
fair_original_size = os.path.getsize("original_weights_only.pkl")

fair_reduction = (1 - quantized_size / fair_original_size) * 100
print("Weight-only 원본 크기(FP64) :", fair_original_size, "bytes")
print("Weight-only 양자화 크기(FP16) :", quantized_size, "bytes")
print("순수 정밀도 효과만으로의 감소율 :", round(fair_reduction, 2), "%")
print("\n(FP64→2바이트인 FP16은 이론상 4배/75% 감소가 기준입니다.
",
      "위 8번의 71~72%는 여기에 sklearn 객체 메타데이터 제거 효과가 조금 섮인 값입니다.)"

### 생각해 볼 질문
1. 이번 실험(random_state=42)에서는 정확도가 전혀 떨어지지 않았습니다. 이게 항상 보장되는 결과일까요, 우연일까요?
2. 정밀도를 FP16보다 더 낮추면(예: INT8) 망은 더 작아지겠지만, 어떤 문제가 생길 수 있을까요?
3. 실제 원본 파일 크기(865B)와 weight-only 크기 중 어떤 걸 기준으로 사용해야 공정한 비교일까요?
4. 이 실습의 결과물(quantized_model.pkl)을 실제 Edge Device에서 쓰려면 추론 코드는 어떻게 바뀌어야 할까요?